# Intro
The philosophy of this class is to create a base object containing the data loaded from a file or from an array. This data is immutable (except for the units). Then, a higher level object is created whose data can be modified (smoothed, truncated, rebinned, etc.) but that can always revert to the original data in the base object if needed (otherwise information is lost when truncating or rebinning). Finally, a visual representation of this higher-level object is created, which contains the PlotItems used for displaying and which may convert the data (for instance if the ViewBox has different units than the spectrum).

In [3]:
# General stuff
import logging
import sys
from pathlib import Path

log = logging.getLogger(__name__)
logging.basicConfig(stream=sys.stdout, level=logging.DEBUG,
    format="%(asctime)s.%(msecs)03d | %(levelname)-8s | %(funcName)s - %(filename)s:%(lineno)d : %(message)s",

                   )
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("PIL").setLevel(logging.WARNING)

In [4]:
from zhunter.spectrum import OneDSpectrum, BaseOneDSpectrum
import os
ZHUNTER_PROJECT_DIR = os.environ.get("ZHUNTER_PROJECT_DIR", Path.cwd().parents[2])

# Base OneDSpectrum
Immutable (except for units) data loaded in memory (ususally from a file)

In [5]:
bspec = BaseOneDSpectrum()

# Data loading
Test loading of data from file, from arrays.

In [6]:
input_dir = ZHUNTER_PROJECT_DIR/"dev/data/test_input_files/"
files = [f for f in input_dir.iterdir() if not f.name.startswith('.')]
fname = files[-3]

In [7]:
bspec.load_from_file(fname)

2024-11-01 10:44:28,159.159 | DEBUG    | read_1D_spectrum - io.py:94 : Read 1D spectrum from file:
/Users/jp279903/Code_projects/zHunter/dev/data/test_input_files/FORS_1D.fits
2024-11-01 10:44:28,160.160 | INFO     | read_fits_1D_spectrum - io.py:392 : Attempting to read file:
/Users/jp279903/Code_projects/zHunter/dev/data/test_input_files/FORS_1D.fits
2024-11-01 10:44:28,166.166 | DEBUG    | get_flux_units - io.py:673 : Using BUNIT: 'erg/cm2/s/A' for flux units
2024-11-01 10:44:28,170.170 | WARNING  | _showwarning - logger.py:235 : UnitsWarning: 'erg/cm2/s/A' contains multiple slashes, which is discouraged by the FITS standard
2024-11-01 10:44:28,172.172 | INFO     | get_wavelength_units - io.py:696 : No unit found in header for wavelength (axis 1), assuming 'nm'
2024-11-01 10:44:28,177.177 | DEBUG    | get_wavelength_constructor - io.py:760 : Using the FITS CD matrix.
2024-11-01 10:44:28,180.180 | DEBUG    | get_wavelength_constructor - io.py:775 : PIX=1.0 VAL=2998.5678710938 DELT=3.

# OneDSpectrum
Higher-level object that can be smoothed, normalized, etc.

In [8]:
spec = OneDSpectrum()
spec.load_from_base_spec(bspec)

2024-11-01 10:44:28,188.188 | DEBUG    | _reset_data - spectrum.py:304 : Resetting data to base spectrum


In [ ]:
spec.set_flux_unit(unit='adu')

2024-11-01 10:44:28,194.194 | DEBUG    | set_flux_unit - spectrum.py:140 : Setting flux units to adu
2024-11-01 10:44:28,197.197 | DEBUG    | _update_units_from_base - spectrum.py:293 : Updating units from base spectrum


# Plotting

I'm thinking about how to represent the spectrum.
In particular, should the units be a part of the `OneDSpectrumVisRep` class or associated with the `ViewBox` instance. Currently I'm leaning towards the `ViewBox` as that would allow a single `OneDSpectrumVisRep` to be represented on multiple `ViewBox` instances (for example for velocity plots).

Edit: This is not possible, a `PlotItem` can only exist on a single `ViewBox`, if you add it to another, it removes it from the first.

In [10]:
from zhunter.spectrum import OneDSpectrumVisRep

In [12]:
sp_vr = OneDSpectrumVisRep(spectrum=spec)

In [13]:
sp_vr.units

{'wvlg': Unit("nm"), 'flux': Unit("adu")}